# 0_3 Boundary conditions and diffraction

**The wave equation tells us which fields are possible. What selects the field we actually observe?**

## Motivation and prerequisites

**Motivation.** We add two waves, compare fields with the same boundary value, and follow a prescribed slit field to a detector. These examples distinguish the equation's possible solutions from the physical conditions that select one.

**Essential prerequisites.** [0_1](0_1_Scalar_Fields_and_Helmholtz.ipynb) and [0_2](0_2_Solutions_of_the_Helmholtz_Equation.ipynb): amplitude, phase, complex fields and possible Helmholtz solutions. No prior knowledge of boundary-value problems or Fourier optics is needed.

**Optional advanced mathematical background.** Linearity, outgoing boundary conditions and Fourier integration justify the examples. The optional far-field derivation explains the sinc pattern; it is not needed to explore the slit.

**First reading.** Observe, predict and adjust each visualization before opening the mathematics. Explain what the aperture prescribes and what propagation calculates. This is part of the 0 series' conceptual overview of scalar diffraction physics; subsequent numbered notebooks revisit the individual concepts and mathematics systematically.

**Next lesson.** [0_4](0_4_Propagation_of_Scalar_Optical_Fields.ipynb) takes a known complex aperture field and follows its propagation. Determining a field from physical material scattering comes later, in 0_5.

We use one deterministic monochromatic scalar field, $u=\operatorname{Re}\{Ue^{-i\omega t}\}$, and relative intensity $|U|^2$. Intensity averages optical cycles without introducing statistical fluctuations. The slit example propagates toward $+z$. Run the cells in order; each explorer has its own reset button.

[Previous: 0_2](0_2_Solutions_of_the_Helmholtz_Equation.ipynb) · [Chapter index](0_0_Foundations_of_Scalar_Optics.ipynb)


:::{note} Interactive examples
Click the **power button** near the top of the page to activate the interactive examples. Binder supplies Python in your browser; no local installation is needed. Wait for the session to connect and the code to finish running, then adjust **Backward-wave amplitude**, **Backward fraction b**, and **Slit width a** in their respective explorers. Use **Reset this example** to restore each starting view. The text, equations and saved figures can be read without activating a session.
:::

For local use, download the {download}`numerical explorer <downloads/grating_physics_numerical_explorer.zip>` or the complete {download}`worked-example scripts <downloads/grating_physics_scripts.zip>` with their shared helpers.

In [ ]:
# Find the shared wave_optics package in the project root.
from pathlib import Path
import sys
import importlib

_support_dir = next((base
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (base / 'wave_optics' / '__init__.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open this notebook within the Grating_Physics folder.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))
# Relocate a cached package without changing the lesson's module reload order.
importlib.invalidate_caches()
if 'wave_optics' in sys.modules:
    sys.modules['wave_optics'].__path__[:] = [str(_support_dir / 'wave_optics')]

import importlib
import numpy as np
import ipywidgets as widgets
from IPython.display import display

if 'scalar_theory_lessons' in globals():
    for old_lesson in scalar_theory_lessons.values():
        old_lesson.close()

# Import normally: restarting the kernel refreshes edited helper modules.
# Explorers are constructed only by the demonstration cells below.
from wave_optics.scalar_theory_view import ScalarTheoryExplorer
from wave_optics.grating_intro import _close_widget_tree

scalar_theory_lessons = {}

def show_scalar_theory(stage):
    if stage in scalar_theory_lessons:
        scalar_theory_lessons[stage].close()
    lesson = ScalarTheoryExplorer(stage)
    scalar_theory_lessons[stage] = lesson
    display(lesson.ui)


## 1. Why can optical waves be added?

Two waves can occupy the same region. At each point, their signed fields can reinforce or oppose one another. The explorer shows a forward wave and a weaker backward wave.

**Predict:** if the backward wave is switched off, should the intensity still vary with position?

**Explore:** set **Backward-wave amplitude** to 0, then restore 0.5. Move **Model time**: which curves change, and which stay fixed?

In [ ]:
show_scalar_theory('superposition')


**Explain:** fields add first; the detector's relative intensity follows from the combined field:

$$\boxed{U=U_1+U_2,\qquad I_{\rm rel}=|U_1+U_2|^2.}$$

The black field curve is the point-by-point sum of the blue and orange curves. The intensity generally differs from the dashed sum of the two separate intensities. When the backward amplitude is zero, the remaining plane wave has uniform intensity.

This works because Helmholtz is **linear**: adding solutions at the same frequency in the same linear homogeneous medium gives another solution. Intensity represents an average over the whole optical cycle, so a single zero crossing does not imply zero intensity. [Notebook 7.1](7_1_Interference_and_Mutual_Coherence_of_Two_Plane_Waves.ipynb) develops interference further.

:::{dropdown} Optional mathematics: linearity and the intensity cross term

For constant complex weights $a,b$ and two solutions with the same $k$,

$$\begin{aligned}
(\nabla^2+k^2)(aU_1+bU_2)
&=a(\nabla^2U_1+k^2U_1)+b(\nabla^2U_2+k^2U_2)\\
&=0.
\end{aligned}$$

Different frequencies can be combined in the time-domain wave equation, but generally need different time factors. For the common-frequency fields here,

$$|aU_1+bU_2|^2=|aU_1|^2+|bU_2|^2
+2\operatorname{Re}\{aU_1(bU_2)^*\}.$$

At the starting amplitudes 1 and 0.5, the intensity ranges from $(1-0.5)^2=0.25$ to $(1+0.5)^2=2.25$. The separate-intensity sum is 1.25. The optional numerical check below uses the original example and verifies the Helmholtz residual; a sampled grid may miss an exact extremum slightly.

:::

In [ ]:
if 'superposition_check_panel' in globals():
    # This notebook owns the panel, output and their layouts.
    _close_widget_tree(superposition_check_panel)

superposition_check = widgets.Output()
with superposition_check:
    k = np.pi
    z = np.linspace(0, 2, 801)
    amplitude1, amplitude2 = 1.0, 0.5
    phase1_deg, phase2_deg = 0.0, 60.0
    U1 = amplitude1*np.exp(1j*(k*z+np.deg2rad(phase1_deg)))
    U2 = amplitude2*np.exp(1j*(-k*z+np.deg2rad(phase2_deg)))
    U = U1+U2
    d2U = -k**2*U1-k**2*U2
    print(f'Maximum Helmholtz residual: {np.max(abs(d2U+k**2*U)):.2e}')
    print(f'Separate-intensity sum: {amplitude1**2+amplitude2**2:.2f}')
    print(f'Sampled intensity range: {np.min(abs(U)**2):.4f} to {np.max(abs(U)**2):.4f}')
    print(f'Exact range: {(amplitude1-amplitude2)**2:.2f} to {(amplitude1+amplitude2)**2:.2f}')
superposition_check_panel = widgets.Accordion(children=[superposition_check], selected_index=None)
superposition_check_panel.set_title(0, "Optional numerical check: superposition and Helmholtz")
display(superposition_check_panel)


## 2. Does one boundary value select one field?

**Observe:** all three cases below can have the same field at $z=0$, yet their waves move differently. Here the boundary is simply the plane where we specify the complex field.

**Predict:** does knowing $U(0)=1$ tell you whether a wave is travelling forward, backward, or standing still?

**Explore:** leave the phase at $0^\circ$ and set **Backward fraction b** to 0, 0.5, and 1. Read the right-hand map upward in time: sloping crests travel; standing-wave nodes stay at fixed positions. Move the time cursor to compare instants.

In [ ]:
show_scalar_theory('boundary')

**Explain:** the equation allows all three fields. A value at one plane alone does not tell us what arrives from either side.

Now specify the experiment: the region is $z>0$, and **no wave arrives from its far end**. This outgoing-wave condition selects the forward wave:

$$\boxed{U(z)=U_{\mathrm{ref}}e^{ikz},\qquad U(0)=U_{\mathrm{ref}}.}$$

The boundary value and this directional condition work together. For curved waves, an outgoing condition similarly selects waves travelling away from a source. Solving a wave equation together with the physical boundary and propagation conditions is called a **boundary-value problem**.

:::{dropdown} Optional mathematics: forward and backward coefficients

The general one-dimensional solution is

$$U''+k^2U=0,\qquad U(z)=B_+e^{ikz}+B_-e^{-ikz}.$$

At the boundary,

$$U(0)=B_++B_-=U_{\mathrm{ref}},\qquad U'(0)=ik(B_+-B_-).$$

The field value alone fixes only a sum. If the derivative is also known,

$$B_+=\frac12\left[U_{\mathrm{ref}}+\frac{U'(0)}{ik}\right],\qquad
B_-=\frac12\left[U_{\mathrm{ref}}-\frac{U'(0)}{ik}\right].$$

The explorer uses

$$\begin{aligned}
U(z)&=e^{i\phi}\left[(1-b)e^{ikz}+be^{-ikz}\right]\\
&=e^{i\phi}\left[\cos(kz)+i(1-2b)\sin(kz)\right],\\
U(0)&=e^{i\phi},\qquad U'(0)=ik(1-2b)e^{i\phi}.
\end{aligned}$$

Thus $b=0$, $1/2$, and $1$ give forward, standing, and backward waves. At $\phi=0$, their real spatial parts coincide, but their imaginary parts and time histories differ. Outgoing propagation into $z>0$ selects $B_-=0$, or $b=0$.

:::

## 3. What happens when a plane wave encounters an aperture?

A plane wave travels along $+z$ toward a long slit of width $a$ at $z=0$. In our thin-mask model, the opening passes the incident field and the blocked region sets it to zero. The slit is uniform along $y$, so we follow a cross-section in $xz$.

**Incident plane wave → slit → transmitted boundary field → propagation → detector intensity.**

**Predict:** when the slit narrows, does the bright region on a distant detector narrow too?

**Explore:** change the width from $20$ to $10\,\mu\mathrm m$, then to $40\,\mu\mathrm m$. Follow the three views. The middle map shows only the first **1 mm** after the slit; the right detector is at **1000 mm**. They show different distances, not adjacent slices.

The finite width is across the slit, not along it: uniformity along y idealizes a slit whose ends are neglected. The x–z image is a cross-section of that model, not the complete diffraction pattern of a finite rectangular hole. [Scalar diffraction optics](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap6_Diffraction/DiffractiveOptics_2022_01Clean.html).


In [ ]:
show_scalar_theory('diffraction')

**Explain:** narrowing the slit broadens the angular diffraction pattern. Fields spreading from different parts of the opening add with different phases at each observation point. Their sum produces bright and dark regions; the space behind the blocked part does not remain dark.

The field we assign immediately after the mask is

$$\boxed{U(x,0)=t(x)U_i(x,0).}$$

Here $t=1$ in the opening and $t=0$ outside. This is a **prescribed thin-mask approximation**: it assigns amplitude and phase at a plane rather than solving electromagnetic scattering at the slit edges. Propagation then determines the field beyond that plane.

The middle map uses outgoing scalar angular-spectrum propagation, including finite-distance structure. Its color scale is fixed relative to the incident intensity. Far away, the simpler **paraxial Fraunhofer approximation** gives the detector curve:

$$\boxed{\frac{I(X)}{I(0)}=\left(\frac{\sin\beta}{\beta}\right)^2,
\qquad \beta=\frac{\pi aX}{\lambda L}.}$$

Here $X$ is position on the detector and $L$ its distance from the slit. The first dark minima are at $X=\pm\lambda L/a$: doubling $a$ halves the central bright width. At reset they are at $\pm30$ mm. Each detector curve uses its **own central intensity**, so compare its width rather than its brightness with the propagation map.

The sinc pattern describes the distant detector; it need not describe a slice near the slit.

:::{dropdown} Optional derivation: from the aperture field to the far-field sinc pattern

Use $\xi$ for a point inside the aperture and $X$ for a detector position. With incident amplitude $A_i$,

$$U_{\mathrm{exit}}(\xi)=U(\xi,0^+)=\begin{cases}A_i,&|\xi|<a/2,\\0,&|\xi|>a/2.\end{cases}$$

The propagation distance is

$$R=\sqrt{L^2+(X-\xi)^2}\simeq L+\frac{X^2}{2L}-\frac{X\xi}{L}+\frac{\xi^2}{2L}.$$

The paraxial approximation requires $|X-\xi|\ll L$. The Fraunhofer limit additionally requires small aperture quadratic phase:

$$\frac{k\xi^2}{2L}\ll1,\qquad \frac{a^2}{\lambda L}\ll1.$$

Dropping that aperture quadratic term, while keeping the detector's quadratic phase, gives

$$U_F(X,L)\propto e^{ikX^2/(2L)}A_i\int_{-a/2}^{a/2}e^{-i(kX/L)\xi}\,d\xi.$$

The omitted propagation factor has constant magnitude across the paraxial detector. With $q=kX/L$ and $\beta=qa/2$,

$$\begin{aligned}
\int_{-a/2}^{a/2}e^{-iq\xi}\,d\xi
&=\left[\frac{e^{-iq\xi}}{-iq}\right]_{-a/2}^{a/2}\\
&=\frac{e^{-iqa/2}-e^{iqa/2}}{-iq}
=a\frac{\sin\beta}{\beta}.
\end{aligned}$$

At $X=0$, the integral is $a$, consistent with $\lim_{\beta\to0}\sin\beta/\beta=1$. Squaring and normalizing to the central value gives the displayed sinc-squared pattern. Its first zeros satisfy

$$\beta=\pm\pi\quad\Longrightarrow\quad X_{\rm zero}=\pm\frac{\lambda L}{a}.$$

For these controls, $a^2/(\lambda L)\leq0.004$ at the detector and $|X|/L\leq0.1$. These conditions support the displayed far-field approximation; they do not hold everywhere in the near-slit map. [TU Delft: Fresnel and Fraunhofer approximations](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap6_Diffraction/DiffractiveOptics_2022_01Clean.html).

**Read the experiment backwards:** if the measured width between the first dark minima is $W$, then $W=2\lambda L/a$ gives $a=2\lambda L/W$. The reset example has $W=60$ mm and $a=20\,\mu\mathrm m$.

:::

:::{dropdown} Optional numerical check and propagation details

The numerical exercise uses the reset values and adds complex contributions across the slit before squaring. It compares the integral divided by $a$ at the centre, halfway to the first zero, at that zero, and beyond it. The amplitudes are $1,\,2/\pi,\,0,\,-2/(3\pi)$, giving intensities about $1,\,0.4053,\,0,\,0.0450$. A negative amplitude is a phase reversal, not a negative intensity.

The propagation map uses the existing scalar angular-spectrum model, uniform along $y$, with the outgoing $+z$ branch and decaying evanescent components. It extends the prescribed field; it does not solve the aperture's material problem. A padded FFT window limits interaction with periodic copies; only the central $160\,\mu\mathrm m$ is displayed. The fixed intensity reference is the incident field, not a power-efficiency calculation. [TU Delft: angular-spectrum propagation](https://qiweb.tudelft.nl/aoi/wavefieldpropagation/wavefieldpropagation/).

:::

In [ ]:
if 'slit_check_panel' in globals():
    _close_widget_tree(slit_check_panel)

slit_check = widgets.Output()
with slit_check:
    lambda_um, a_um, L_mm = 0.6, 20.0, 1000.0
    k = 2*np.pi/lambda_um
    first_zero_mm = lambda_um*L_mm/a_um
    X_mm = first_zero_mm*np.array([0.0, 0.5, 1.0, 1.5])
    sample_count = 12000
    xi_um = -a_um/2+(np.arange(sample_count)+0.5)*a_um/sample_count
    amplitude_sum = np.mean(np.exp(-1j*k*(X_mm[:, None]/L_mm)*xi_um), axis=1)
    beta = np.pi*a_um*X_mm/(lambda_um*L_mm)
    # NumPy sinc(s) means sin(pi*s)/(pi*s), including its value 1 at s=0.
    analytic_amplitude = np.sinc(beta/np.pi)
    print('X (mm)   summed amplitude   analytic amplitude   normalized intensity')
    for X, summed, exact in zip(X_mm, amplitude_sum, analytic_amplitude):
        print(f'{X:6.1f}      {summed.real:+.6f}            {exact:+.6f}               {abs(summed)**2:.6f}')
    print(f'Maximum complex-amplitude mismatch: {np.max(abs(amplitude_sum-analytic_amplitude)):.2e}')
slit_check_panel = widgets.Accordion(children=[slit_check], selected_index=None)
slit_check_panel.set_title(0, "Optional numerical check: add aperture contributions")
display(slit_check_panel)


(grating-field-flow)=

## 4. How does this connect to a grating?

Repeating the slit produces a grating. We still need a complex field at an output plane before we can predict what propagates beyond it.

**Predict:** if you already know that field's amplitude and phase, do you need to solve the material problem again just to identify its wave components?

Follow the two routes below. Here the grating repeats every $d$ along $x$, has no $y$ variation, and receives a monochromatic plane wave. We now adopt the later grating lessons' transmission toward **$z<0$**, with $z=0$ at the output reference plane.

**Physical grating and incident field:** refractive-index geometry $n(\mathbf r)$ and illumination $U_i(\mathbf r)$

| ↓ Prescribed thin-mask route | ↓ Scalar boundary-value route |
| --- | --- |
| Choose an approximate complex transmission $t(x)$. | Use the refractive-index geometry $n(\mathbf r)$. |
| Prescribe $U_{\mathrm{exit}}(x)=t(x)U_i(x,0)$. | Solve $\nabla^2U+k_0^2n^2(\mathbf r)U=0$ with the illumination, appropriate interface conditions and outgoing conditions for the scattered field. |
| ↓ Assigned output field | ↓ Output field from the solution |

**Either route → known complex output-plane field → Fourier decomposition → coefficients $C_m$ → order fields $U_m$.**

**Explain:** Fourier decomposition rewrites a known field as a sum of simpler spatial oscillations. Their spatial frequencies describe how rapidly phase varies across the plane. Their amplitudes and phases reconstruct the field on the plane; in a homogeneous output region, their wavevectors determine how they propagate. A periodic grating selects discrete components called **diffraction orders**.

**Obtaining the field and decomposing it are different tasks.** A boundary-value model determines a field from the physical configuration. Fourier decomposition extracts components from an already known field; even an exact decomposition cannot correct an approximate mask model.

$U_{\mathrm{exit}}(x)=U(x,0^-)$ is the **boundary trace**: the complex field restricted to one plane, not the full spatial Helmholtz solution. The mask prescribes this trace; a boundary-value calculation obtains it from its solution. With the outgoing/decaying choice, suitable plane data can be propagated into the homogeneous output region.

Each propagating order needs both a wavevector and a complex amplitude:

$$\boxed{U_m(\mathbf r)=C_m e^{i\mathbf k_m\cdot\mathbf r},\qquad C_m=A_m e^{i\phi_m}.}$$

The wavevector sets direction and spatial phase variation; $C_m$ sets magnitude and phase offset at the chosen reference. **This is a preview:** calculating $C_m$ belongs to a later lesson.

:::{dropdown} Optional mathematics: coefficients, Floquet condition and reference plane

Here $k_0=2\pi/\lambda_0$, with vacuum wavelength $\lambda_0$; in a homogeneous medium, $k=nk_0=2\pi/\lambda$. A known field includes amplitude and phase, whether prescribed, measured/reconstructed, or calculated from a boundary-value problem. Different models may give different traces and therefore different coefficients.

For plane-wave illumination, the trace has the Floquet (Bloch-periodic) property

$$U_{\mathrm{exit}}(x+d)=e^{ik_{x,i}d}U_{\mathrm{exit}}(x),\qquad
k_{x,m}=k_{x,i}+\frac{2\pi m}{d},\quad m\in\mathbb Z.$$

The field repeats up to an incident phase step. If it is square-integrable over one period, its coefficients at $z=0$ are

$$C_m=\frac1d\int_0^d U_{\mathrm{exit}}(x)e^{-ik_{x,m}x}\,dx.$$

The factor $1/d$ retains field-amplitude units. $A_m=|C_m|\geq0$, and $\phi_m=\arg C_m$ when $C_m\ne0$. A complete expansion can also contain evanescent orders, which decay into the output region rather than propagate away.

For another reference plane in the same homogeneous output region,

$$U_m(x,z)=C_m(z_{\rm ref})e^{ik_{x,m}x}e^{ik_{z,m}(z-z_{\rm ref})},\qquad
C_m(z_{\rm ref})=C_m(0)e^{ik_{z,m}z_{\rm ref}}.$$

For a thin mask, $C_m(0)=A_i t_m$: $t_m$ is a dimensionless Fourier coefficient of $t(x)$ and $A_i$ is the incident amplitude at the same reference. These field coefficients are not power fractions; diffraction efficiency requires incident-power and normal-flux normalization, which we leave to later lessons.

The Fourier representation is exact for an appropriate periodic or Bloch-periodic field in the Fourier-series sense. Sampling and truncation add numerical errors; neither decomposition nor propagation independently enforces the grating's material conditions. [Maystre, §2.3](https://www.fresnel.fr/files/gratings/Second-Edition/GratingsTheoryandNumericApplicationsSecondEdition.pdf#page=43).

:::

**Notation:** use $U_{\mathrm{exit}}$ at the exit plane, $U_{\mathrm{ref}}$ at a general reference plane, and $C_m$ (or $C_{mn}$) for field coefficients. [Shared convention](0_0_Foundations_of_Scalar_Optics.ipynb#exit-field-notation).


## Checkpoint: explain the pictures in words

1. Why do we add fields before calculating intensity?
2. How can forward, backward and standing waves share the same boundary value? What selects the forward wave?
3. What does the thin mask prescribe, and what does propagation calculate?
4. Why does a narrower slit produce a broader distant pattern? Must the near-slit pattern have the same shape?
5. What is the difference between determining a field and decomposing a known field?
6. Why does an order need a complex amplitude as well as a direction?

:::{dropdown} Answers

- Linearity applies to fields; squaring their sum includes their interaction. One boundary value does not specify the wave's direction; the absence of a wave arriving from the far side selects outgoing propagation.
- The mask assigns amplitude and phase on a plane; propagation extends that field into the output region. A narrower slit requires a broader spread of propagation directions, giving a wider distant pattern.
- Near the slit, phase contributions have not reached the far-field limit, so the pattern can differ. A physical model determines the field, while Fourier decomposition expresses a known field in components. Each component needs a magnitude and phase offset as well as a wavevector.

:::

Continue to [Notebook 0_4: Propagation of scalar optical fields](0_4_Propagation_of_Scalar_Optical_Fields.ipynb) to follow a known field from one plane to another.

## Worked-out examples

These optional scripts show the numerical implementation of the topics in this notebook.
Use the complete {download}`scripts download <downloads/grating_physics_scripts.zip>` to keep the plotting helper and
`wave_optics` package with the examples. Edit **INPUT PARAMETERS**, then run the file
or execute its `# %%` sections in order. Read **CHECK AND INTERPRET** with the plots.

### 1. Why can optical waves be added?

**Script:** [0_3_01_superposition_residual.py](Wave_solver_py_scripts/scripts/chapter_0/0_3_01_superposition_residual.py)

Check linear superposition with complex amplitudes.

### 2. Does one boundary value select one field?

**Script:** [0_3_02_boundary_nonuniqueness.py](Wave_solver_py_scripts/scripts/chapter_0/0_3_02_boundary_nonuniqueness.py)

Compare fields that share the same boundary value.

### 3. What happens when a plane wave encounters an aperture?

**Script:** [0_3_03_slit_phasor_quadrature.py](Wave_solver_py_scripts/scripts/chapter_0/0_3_03_slit_phasor_quadrature.py)

Integrate a slit far field and recover the sinc pattern.

### 4. How does this connect to a grating?

**Script:** [0_3_04_periodic_boundary_field.py](Wave_solver_py_scripts/scripts/chapter_0/0_3_04_periodic_boundary_field.py)

Prescribe an illuminated grating boundary and test its translation.


## References and conventions

A. P. Konijnenberg, A. J. L. Adam, and H. P. Urbach, [Basic Electromagnetic and Wave Optics, TU Delft](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap1_Basics/Basics_2022_01Clean.html): harmonic fields and the scalar wave equation. TU Delft, [Wavefield propagation](https://qiweb.tudelft.nl/aoi/wavefieldpropagation/wavefieldpropagation/): the angular-spectrum representation and outgoing/evanescent components. TU Delft, [Scalar diffraction theory](https://qiweb.tudelft.nl/aoi/scalardiffractiontheory/scalardiffractiontheory/): prescribed boundary fields and propagation. A. P. Konijnenberg, A. J. L. Adam, and H. P. Urbach, [Scalar Diffraction Optics, TU Delft](https://interactivetextbooks.tudelft.nl/tn2421/content/Chap6_Diffraction/DiffractiveOptics_2022_01Clean.html): Fresnel and Fraunhofer approximations. D. Maystre, [Gratings: Theory and Numeric Applications, §2.3](https://www.fresnel.fr/files/gratings/Second-Edition/GratingsTheoryandNumericApplicationsSecondEdition.pdf#page=43): optional background for the grating-order preview.

Throughout, $\lambda$ is wavelength in the medium and the time convention is $e^{-i\omega t}$. The pictures illustrate deterministic prescribed scalar fields. They do not calculate electromagnetic scattering by the aperture or statistical coherence.

## Selected freely accessible reading

Optional reading: start with the suggested passage. Freely accessible works are not necessarily in the public domain.

- **M. Mansuripur. [A Tutorial on the Classical Theories of Electromagnetic Scattering and Diffraction](https://arxiv.org/abs/2009.06706).** Nanophotonics 10, 315–342 (2021); arXiv version submitted 2020. **Access:** Free arXiv manuscript; not public domain. **Read for:** Optional advanced reading: scalar diffraction and its electromagnetic assumptions.

- **C. Huygens. [Traité de la lumière / Treatise on Light](https://www.gutenberg.org/cache/epub/14725/pg14725-images.html).** 1690; English translation by S. P. Thompson (1912). **Access:** Historical treatise: original marked Public Domain; translation public domain in the USA. **Read for:** Read Chapter I for historical wavefront construction, not modern field theory.

[Source catalogue and access notes](supplement/reading_sources.json).
